In [1]:
%pip install "pandas>=2.0" numpy "scikit-learn>=1.2"

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 8.2/8.2 MB 43.0 MB/s  0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -------------- ------------------------- 13.1/36.6 MB 63.0 MB/s eta 0:00:01
   ---------------------------- ----------- 26.5/36.6 MB 61.9 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 61.6 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 43.7 MB/s  0:00:00

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ ---------------


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path("experimental_protocol")
OUT = Path("technical_tests")
OUT.mkdir(exist_ok=True)
protocol = json.loads((ROOT / "protocol.json").read_text(encoding="utf-8"))
folds = json.loads((ROOT / "folds.json").read_text(encoding="utf-8"))
source = Path("sabrina_original.csv")
assert hashlib.sha256(source.read_bytes()).hexdigest() == protocol["source_sha256"], "CSV mudou: revisar protocolo."
FEATURES = protocol["features"]
TARGET = protocol["target"]
checks = []

def passed(name):
    checks.append({"teste": name, "status": "OK"})


usecols = ["track_id", "album"] + [c for c in FEATURES if c != "duration_min"] + ["duration_ms", TARGET]
raw = pd.read_csv(source, sep=";", usecols=usecols)
raw["album_family"] = raw["album"].replace({"emails i can’t send fwd": "emails i can’t send"})
reserved_ids = set(raw.loc[raw.album_family.eq(protocol["test_family"]), "track_id"])
dev = raw.loc[~raw.album_family.eq(protocol["test_family"])].copy()
del raw  # trabalhar so no desenvolvimento daqui em diante.
dev[TARGET] = pd.to_numeric(dev[TARGET].astype(str).str.replace(".", "", regex=False), errors="raise")
dev["instrumentalness"] = pd.to_numeric(dev["instrumentalness"].astype(str).str.replace(",", ".", regex=False), errors="raise")
dev["duration_min"] = dev["duration_ms"] / 60000
dev = dev.set_index("track_id", verify_integrity=True)
assert len(dev) == protocol["n_development"]
assert len(reserved_ids) == protocol["n_test"]
assert set(dev.index).isdisjoint(reserved_ids)
assert all(pd.api.types.is_numeric_dtype(dev[c]) for c in FEATURES)
assert not np.isinf(dev[FEATURES].to_numpy(dtype=float)).any()
assert dev[TARGET].notna().all() and (dev[TARGET] >= 0).all()
assert not set([TARGET, "log_streams", "track_id", "album_family"]) & set(FEATURES)
passed("Integridade da base, tipos, alvo e isolamento do teste")

# primeiro fold fixado
f = folds[0]
train = dev.loc[f["train_ids"]].copy()
val = dev.loc[f["validation_ids"]].copy()
assert set(train.album_family).isdisjoint(set(val.album_family))
assert set(train.index).isdisjoint(set(val.index))
assert set(train.index) | set(val.index) == set(dev.index)
passed("Separação de grupos do fold técnico")

small = train.iloc[:24]
X_small, y_small = small[FEATURES], small[TARGET]
X_check, y_check = val[FEATURES], val[TARGET]
assert len(X_small) == len(y_small) and X_small.shape[1] == 10
print("Treino técnico:", len(small), "faixas; validação técnica:", len(val), "faixas")
print("Nenhuma faixa do teste final está nesses conjuntos.")

Treino técnico: 24 faixas; validação técnica: 20 faixas
Nenhuma faixa do teste final está nesses conjuntos.


C:\Users\gusta\AppData\Local\Temp\ipykernel_9500\2760222189.py:43: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  dev = dev.set_index("track_id", verify_integrity=True)


In [3]:
def make_models():
    models = {
        "baseline": Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", DummyRegressor(strategy="mean")),
        ]),
        "linear": Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("model", LinearRegression()),
        ]),
    }
    for seed in protocol["rf_seeds"]:
        models[f"forest_seed_{seed}"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", RandomForestRegressor(
                **protocol["rf_params"], random_state=seed, n_jobs=-1
            )),
        ])
    return {
        name: TransformedTargetRegressor(
            regressor=pipe, func=np.log1p, inverse_func=np.expm1, check_inverse=True
        )
        for name, pipe in models.items()
    }

models = make_models()
assert len(models) == 5  # Baseline, linear e três sementes da mesma floresta.
passed("Construção das três classes de modelo previstas")
print("Pipelines construídos; sem fit até este ponto.")

Pipelines construídos; sem fit até este ponto.


In [4]:
fitted = {}
for name, model in models.items():
    fitted[name] = clone(model).fit(X_small, y_small)
    fitted_pipe = fitted[name].regressor_
    Z_train = fitted_pipe[:-1].transform(X_small)
    Z_val = fitted_pipe[:-1].transform(X_check)
    assert Z_train.shape == X_small.shape
    assert Z_val.shape == X_check.shape
    assert np.isfinite(Z_train).all() and np.isfinite(Z_val).all()
    imputer = fitted_pipe.named_steps["imputer"]
    assert np.allclose(imputer.statistics_, X_small.median().to_numpy())
    if "scaler" in fitted_pipe.named_steps:
        imputed_train = imputer.transform(X_small)
        assert np.allclose(fitted_pipe.named_steps["scaler"].mean_, imputed_train.mean(axis=0))
    p = fitted[name].predict(X_check)
    p_log = fitted_pipe.predict(X_check)
    assert p.shape == (len(y_check),)
    assert np.isfinite(p).all() and np.isfinite(p_log).all()
    assert (p >= 0).all(), "Previsão negativa: investigar antes de experimentar."
    assert np.allclose(p, np.expm1(p_log))
    scores = [mean_absolute_error(y_check, p),
              np.sqrt(mean_squared_error(y_check, p)), r2_score(y_check, p),
              mean_absolute_error(np.log1p(y_check), p_log)]
    assert np.isfinite(scores).all()
    again = clone(model).fit(X_small, y_small).predict(X_check)
    assert np.allclose(p, again, rtol=1e-10, atol=1e-6)
    passed(f"{name}: fit, pré-processamento, escalas, previsões, métricas e reprodução")

expected = np.expm1(np.log1p(y_small).mean())
assert np.allclose(fitted["baseline"].predict(X_check), expected)
passed("Baseline coincide com a média do log do treinamento invertida")

X_missing = X_small.copy()
X_missing.iloc[0, 0] = np.nan
missing_model = clone(models["linear"]).fit(X_missing, y_small)
assert np.isfinite(missing_model.regressor_[:-1].transform(X_missing)).all()
assert np.isfinite(missing_model.predict(X_check)).all()
assert X_small.iloc[0, 0] == small[FEATURES].iloc[0, 0]
passed("Imputação de ausência artificial sem alterar os dados originais")

forest_logs = np.vstack([fitted[f"forest_seed_{seed}"].regressor_.predict(X_check)
                         for seed in protocol["rf_seeds"]])
ensemble_prediction = np.expm1(forest_logs.mean(axis=0))
assert ensemble_prediction.shape == (len(y_check),)
assert np.isfinite(ensemble_prediction).all() and (ensemble_prediction >= 0).all()
passed("Agregação das sementes da floresta na escala prevista pelo protocolo")
print("Testes de execução concluídos. Métricas técnicas calculadas, mas não usadas para seleção.")

Testes de execução concluídos. Métricas técnicas calculadas, mas não usadas para seleção.


In [5]:
probe = TransformedTargetRegressor(
    regressor=Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("model", DecisionTreeRegressor(random_state=2026)),
    ]), func=np.log1p, inverse_func=np.expm1
)
probe.fit(X_small, y_small)
train_log = probe.regressor_.predict(X_small)
val_log = probe.regressor_.predict(X_check)
train_mae = mean_absolute_error(np.log1p(y_small), train_log)
val_mae = mean_absolute_error(np.log1p(y_check), val_log)
assert not X_small.duplicated().any(), "Teste de memorização exige entradas distintas."
assert train_mae < 1e-8, "Investigar falha de memorização no subconjunto."
assert np.isfinite(val_mae)
passed("Árvore-sonda consegue memorizar o subconjunto com entradas distintas")
diagnostic = pd.DataFrame({
    "conjunto": ["treino técnico", "validação técnica"],
    "n": [len(y_small), len(y_check)],
    "MAE_log1p_arvore_sonda": [train_mae, val_mae]
})
print(diagnostic.to_string(index=False))
print("Diagnóstico ilustrativo: não comparar candidatos nem escolher configurações com esta tabela.")

         conjunto  n  MAE_log1p_arvore_sonda
   treino técnico 24                0.000000
validação técnica 20                2.324821
Diagnóstico ilustrativo: não comparar candidatos nem escolher configurações com esta tabela.


In [ ]:
results = pd.DataFrame(checks)
assert results["status"].eq("OK").all()
results.to_csv(OUT / "checks.csv", index=False)
diagnostic.to_csv(OUT / "diagnostic.csv", index=False)
metadata = {
    "source_sha256": protocol["source_sha256"],
    "protocol_version": protocol["version"],
    "python": platform.python_version(), "pandas": pd.__version__,
    "numpy": np.__version__, "scikit_learn": sklearn.__version__,
    "technical_fold": {"split_seed": f["split_seed"], "fold": f["fold"]},
    "small_train_ids": small.index.tolist(), "validation_ids": val.index.tolist(),
    "n_checks": len(results), "all_passed": True,
    "test_evaluated": False,
    "purpose": "testes técnicos; não seleção experimental"
}
(OUT / "execution_register.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print(results.to_string(index=False))
print(f"\n{len(results)} verificações aprovadas. Teste final não avaliado.")

                                                                            teste status
                           Integridade da base, tipos, alvo e isolamento do teste     OK
                                              Separação de grupos do fold técnico     OK
                                  Construção das três classes de modelo previstas     OK
                                  Construção das três classes de modelo previstas     OK
      baseline: fit, pré-processamento, escalas, previsões, métricas e reprodução     OK
        linear: fit, pré-processamento, escalas, previsões, métricas e reprodução     OK
forest_seed_11: fit, pré-processamento, escalas, previsões, métricas e reprodução     OK
forest_seed_29: fit, pré-processamento, escalas, previsões, métricas e reprodução     OK
forest_seed_47: fit, pré-processamento, escalas, previsões, métricas e reprodução     OK
                    Baseline coincide com a média do log do treinamento invertida     OK
                  Imp